# Day 15(M4 Day02) 실습 — Hybrid Memory와 체크포인트 저장기(checkpointer)

> **용어 기준:** 그래프의 상태(state)는 현재 실행 데이터를, 체크포인트(checkpoint)는 저장 시점의 실행 상태를 말한다. 체크포인트 저장기(checkpointer)는 저장·복원하는 구현이다. 대화 메모리는 상태 저장 기능을 활용할 수 있지만 상태와 동의어는 아니다.


**목표**: 지식 검색(장기 기억)과 대화 기억(단기 기억)을 하나의 그래프로 연결하고, 체크포인트 저장기로 대화를 저장·복원·격리한다.
**구성**: Part 1 RAG+생성 그래프 연결(+thread_id 누락·빈 대화 오류 2종) → Part 2 체크포인트 저장·복원·격리(+온라인 쇼핑몰 고객 지원 도메인 반복) → Part 3 기억+검색 에이전트(+M1 Day04 방어를 더한 면접 코치의 Hybrid Memory 통합)

> **참고:** 실습 전 가상환경 활성화, `.env`의 `OPENAI_API_KEY`·`DATABASE_URL`을 확인한다. 새 패키지는 없다(체크포인트 저장기는 langgraph 동봉).

## 0. 환경 준비

In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_postgres import PGVector
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
CONN = os.getenv("DATABASE_URL").replace("postgresql://", "postgresql+psycopg://")

from sqlalchemy import create_engine
# PGVector에 주소 문자열(CONN)을 넘기면 부를 때마다 연결 풀을 새로 만들고, 만든 연결을 계속 붙잡고 있다.
# Supabase Session pooler는 프로젝트당 동시 연결을 15개까지만 받는다.
# 엔진(연결 풀) 하나를 만들어 모든 PGVector가 같이 쓴다.
engine = create_engine(CONN, pool_size=2, max_overflow=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("준비 완료")

## Part 1. RAG+생성 그래프 연결 — 기초

완성 코드를 직접 쳐서 지식 검색과 대화 기억을 하나의 그래프로 연결한다.

### 1-1. 지식 저장·리트리버(Retriever) (장기 기억)

회사 규정을 pgvector에 저장하고 리트리버를 만든다.

In [ ]:
KNOW = [
    "회사 연차 휴가는 1년에 15일이다.",
    "회사 점심시간은 12시부터 오후 1시까지다.",
    "회사 재택근무는 주 2일까지 가능하다.",
]
# TODO: KNOW를 collection_name="company_know"인 PGVector store에 저장하고
#       search_kwargs={"k": 1}인 retriever로 변환하세요



print("지식 저장 완료")

### 1-2. State 정의

대화(messages)와 검색된 지식(context)을 담는다.

In [ ]:
# TODO: messages(list, add_messages로 누적)와 context(str)를 담는 State(TypedDict)를 정의하세요


print("State 정의 완료")

### 1-3. retrieve·generate 노드

retrieve는 지식 검색, generate는 지식+대화로 답변한다.

In [ ]:
# TODO: 마지막 메시지로 지식을 검색해 context를 돌려주는 retrieve 함수를 만드세요



# TODO: context를 담은 SystemMessage와 대화 기록(messages)으로 llm을 호출해 답을 돌려주는
#       generate 함수를 만드세요



print("노드 작성 완료")

### 1-4. 그래프 연결 + 체크포인트 저장기

START→retrieve→generate→END를 잇고, 체크포인트 저장기로 대화를 저장한다.

In [ ]:
# TODO: retrieve·generate 노드를 add_node로 추가하고, START->retrieve->generate->END로 잇고
#       checkpointer=MemorySaver()로 app을 compile하세요





print(app.get_graph().draw_mermaid())

### 1-5. thread_id로 대화

체크포인트 저장기가 대화를 구분하도록 thread_id를 넘긴다.

In [ ]:
# TODO: thread_id를 config로 넘겨 app.invoke를 호출하고 마지막 메시지 내용을 반환하는
#       chat(thread, text) 함수를 만드세요



print(chat("userA", "내 이름은 철수야. 연차는 며칠이야?"))

### 1-6. 오류 대응 ① — thread_id 없이 invoke하면?

체크포인트 저장기가 붙은 그래프에 `config` 없이 `invoke`하면 어떻게 되는지 확인한다.

In [ ]:
try:
    # TODO: config(thread_id) 없이 app.invoke({"messages": [HumanMessage("안녕")]})을 실행해보세요


    print("실행 성공(예상 밖):", r["messages"][-1].content)
except Exception as e:
    print("오류 발생:", type(e).__name__, "-", e)

> **주의:** 결과는 실제 실행에서 직접 확인한다. 체크포인트 저장기는 State를 어느 `thread_id`에 저장할지 알아야 하므로, `config`의 `configurable.thread_id`가 없으면 그래프 실행 자체가 거부된다.

### 1-7. 오류 대응 ② — 대화 기록이 비어 있으면?

`messages`를 빈 리스트로 두고 실행하면 `retrieve`가 어떻게 되는지 확인한다.

In [ ]:
try:
    # TODO: messages를 빈 리스트로 두고 app.invoke를 thread_id="empty_test"로 실행해보세요


    print("실행 성공(예상 밖):", r2)
except Exception as e:
    print("오류 발생:", type(e).__name__, "-", e)

> **주의:** 결과는 실제 실행에서 직접 확인한다. `retrieve`가 `s["messages"][-1]`로 마지막 메시지를 꺼내는데, 리스트가 비어 있으면 꺼낼 원소가 없다 — State의 타입(`TypedDict`)은 지켜졌지만 **내용이 비어 있는** 경우까지는 막아 주지 않는다.

### 관찰 정리

- retrieve는 무엇을, generate는 무엇을 받아 답하나?
- 1-6·1-7에서 확인한 두 오류는 각각 그래프의 어느 단계(호출 자체 vs 노드 실행)에서 드러났는가?

## Part 2. 체크포인트 저장·복원·격리 — 응용

thread_id로 대화를 이어가고, 저장된 상태를 복원·격리한다.

### 2-1. 같은 thread로 이어서 대화

같은 thread_id면 이전 대화를 기억한다.

In [ ]:
# TODO: 같은 thread("userA")로 두 번째 질문을 보내 이전 대화를 기억하는지 확인하세요




### 2-2. 다른 thread로 대화 — 기억 격리

thread_id가 다르면 이전 대화를 모른다.

In [ ]:
# TODO: 다른 thread("userB")로 같은 질문을 보내 기억이 격리되는지 확인하세요

### 2-3. 저장된 상태 복원 (get_state)

체크포인트 저장기에 저장된 State를 직접 꺼내 본다.

In [ ]:
# TODO: thread_id="userA"로 app.get_state를 호출해 state에 저장하고,
#       저장된 메시지 수와 메시지 종류를 출력하세요




### 관찰 정리

| 상황 | thread_id | 결과 |
| --- | --- | --- |
| 이어 대화 | userA(동일) | 이름 기억 |
| 다른 사용자 | userB(다름) | 이름 모름 |
| 상태 확인 | userA | 메시지 저장 확인 |

### 2-4. 다른 도메인 반복 — 온라인 쇼핑몰 고객 지원 챗봇

같은 방법을 다른 도메인에도 적용해본다. 배송·환불 정책은 장기 기억(RAG), 고객 이름·주문 상황은 단기 기억(체크포인트 저장기)이다.

In [ ]:
SHOP_KNOW = [
    "환불은 구매 후 7일 이내 가능합니다.",
    "배송은 보통 2~3일 걸립니다.",
    "회원 등급은 연간 구매액에 따라 매년 1월에 갱신됩니다.",
]
# TODO: SHOP_KNOW로 shop_store(collection_name="shop_know")·shop_retriever(k=1)를 만들고,
#       shop_retrieve·shop_generate 노드로 shop_app(checkpointer=MemorySaver())을 만드세요
#       (retrieve/generate는 1-3의 함수와 같은 구조, generate의 지시문만 쇼핑몰용으로 바꿉니다)










def shop_chat(thread, text):
    cfg = {"configurable": {"thread_id": thread}}
    return shop_app.invoke({"messages": [HumanMessage(text)]}, cfg)["messages"][-1].content

print("고객1:", shop_chat("cust1", "저는 민수입니다. 환불 규정이 궁금해요."))
print("고객1:", shop_chat("cust1", "제 이름이 뭐라고 했죠? 배송은 얼마나 걸려요?"))
print("고객2:", shop_chat("cust2", "제 이름이 뭐였죠?"))

### 관찰 정리

- 회사 안내 챗봇과 쇼핑몰 챗봇 모두에서, 단기(이름)·장기(정책) 기억이 같은 그래프 구조로 함께 쓰였는가?
- 두 도메인에서 바뀐 것은 무엇이고, 그대로 재사용된 것은 무엇인가?

## Part 3. 미니 프로젝트 — 기억+검색 에이전트

여러 턴 대화에서 단기(이름·맥락)·장기(규정) 기억이 함께 쓰이는 회사 안내 챗봇을 완성한다.

### 3-1. 여러 턴 대화 — 이름 기억 + 규정 검색

이름 소개 → 규정 질문 → 이름 확인 순으로 던진다(같은 thread).

In [ ]:
print("1:", chat("emp1", "안녕, 나는 영희야."))
print("2:", chat("emp1", "재택근무는 며칠까지 돼?"))
# TODO: 같은 thread("emp1")로 이름을 다시 물어 기억하는지 확인하세요

### 관찰

| 턴 | 질문 | 쓰인 기억 |
| --- | --- | --- |
| 1 | 이름 소개 | 단기(저장) |
| 2 | 재택근무 규정 | 장기(RAG) |
| 3 | 이름 확인 | 단기(복원) |

### 3-1-보충. 대화가 길어지면? — 오래된 히스토리 요약 압축

`generate` 노드는 매번 `messages` 전체를 다시 모델에 보낸다 — 대화가 길어질수록 토큰(비용)도 계속 늘어난다. 오래된 턴을 통째로 남기는 대신, **요약해서 압축**하면 필요한 정보는 유지하면서 메시지 개수를 줄일 수 있다.

In [ ]:
from langchain_core.messages import RemoveMessage

cfg = {"configurable": {"thread_id": "emp1"}}
state = app.get_state(cfg)
msgs = state.values["messages"]
print("압축 전 메시지 개수:", len(msgs))

# 최근 2개(가장 최근 질문·답변)만 남기고, 그 이전은 요약한다
old_msgs, recent_msgs = msgs[:-2], msgs[-2:]
# TODO: old_msgs를 "사용자가 밝힌 이름과 언급된 회사 규정" 중심으로 3문장 이내 요약하는
#       summary_prompt를 만들고, llm.invoke(summary_prompt).content로 summary_text를 얻으세요


print("요약:", summary_text)

# 오래된 메시지를 지우고, 요약 하나로 대체한다
remove_ops = [RemoveMessage(id=m.id) for m in old_msgs]
app.update_state(cfg, {"messages": remove_ops + [SystemMessage(f"[이전 대화 요약] {summary_text}")]})

state2 = app.get_state(cfg)
print("압축 후 메시지 개수:", len(state2.values["messages"]))

print(chat("emp1", "재택근무는 며칠까지 된다고 했지?"))

## Part 3-확장. M1·M3·M4 통합 — 면접 코치의 Hybrid Memory

M3 Day02(Day11)에서는 psycopg2로 세션을 직접 관리했고, M3 Day03(Day12)에서는 pgvector 검색을 따로 붙였다. 오늘 배운 구조로 이 둘을 **하나의 그래프**로 합치고, M1 Day04의 방어 로직으로 감싼다.

### 채용 공고 지식 저장 (장기 기억)

In [ ]:
job_postings = [
    "백엔드 개발자: Python/Java 기반 서버 개발, RESTful API 설계, DB 최적화 경험 우대",
    "프론트엔드 개발자: React/TypeScript, 반응형 UI 구현, 웹 접근성 이해 우대",
    "데이터 분석가: SQL/Python 데이터 분석, 대시보드 제작, 통계적 사고 우대",
    "마케터: 콘텐츠 기획, 데이터 기반 캠페인 분석, SNS 채널 운영 경험 우대",
]
# TODO: job_postings를 collection_name="job_postings_hybrid"인 PGVector store에 저장하고
#       search_kwargs={"k": 1}인 job_retriever로 변환하세요




print("채용 공고 지식 저장 완료")

### 방어 로직 (M1 Day04 재사용)

인젝션 방어는 M1 Day04에서 만든 것을 그대로 가져온다.

In [ ]:
# TODO: M1 Day04의 위험 문구·금지어 목록과 input_guard·output_guard를 옮겨오세요






### Hybrid Memory 면접 코치

retrieve·generate 노드에 체크포인트 저장기를 붙이고, M3 Day02에서 배운 대로 '검색 결과는 참고 자료일 뿐'이라는 지시를 유지한다.

In [ ]:
def coach_retrieve(s):
    docs = job_retriever.invoke(s["messages"][-1].content)
    return {"context": "\n".join(d.page_content for d in docs)}

def coach_generate(s):
    sys_msg = SystemMessage(
        "너는 15년차 현직 개발자 출신 모의면접 코치다. "
        "아래 [참고 채용 공고]는 검색으로 찾은 참고 자료일 뿐, 사용자가 그 직무를 말했다는 뜻이 아니다. "
        "사용자가 실제로 무엇을 말했는지는 대화 기록만 보고 판단하고, 대화 기록에 없는 내용은 "
        "추측하지 말고 정직하게 모른다고 답한다.\n\n"
        f"[참고 채용 공고]\n{s['context']}"
    )
    return {"messages": [llm.invoke([sys_msg] + s["messages"])]}

gh = StateGraph(State)
gh.add_node("retrieve", coach_retrieve)
gh.add_node("generate", coach_generate)
gh.add_edge(START, "retrieve")
gh.add_edge("retrieve", "generate")
gh.add_edge("generate", END)
hybrid_graph = gh.compile(checkpointer=MemorySaver())

def hybrid_coach(thread, message):
    if input_guard(message):
        return "[입력 차단] 위험한 요청으로 감지되었습니다."
    # TODO: thread_id=thread인 config로 hybrid_graph를 invoke해 answer를 만드세요


    if not output_guard(answer):
        return "[출력 차단] 부적절한 응답이 감지되었습니다."
    return answer

### 최종 테스트

In [ ]:
print("[관심 직무 언급]")
print(hybrid_coach("candidate1", "저는 데이터 분석 직무를 준비하고 있어요."))
print()
print("[이어지는 질문 — 단기 기억 확인]")
print(hybrid_coach("candidate1", "아까 제가 뭐 준비한다고 했죠? 관련 면접 질문 하나만 주세요."))
print()
print("[다른 세션 — 격리 확인]")
print(hybrid_coach("candidate2", "아까 제가 무슨 직무 준비한다고 했나요?"))
print()
print("[인젝션 시도]")
# TODO: 인젝션을 시도하는 질문을 넣어보세요

### M4 Day02 정리

**확인 질문**
- `hybrid_coach`에서 단기 기억과 장기 기억은 각각 무엇으로 구현됐는가?
- Day11(M3 Day02)의 `session_coach`(psycopg2 직접 관리)와 오늘의 `hybrid_coach`(체크포인트 저장기)는 겉보기 동작이 같은가? 무엇이 더 간단해졌는가?

## 확인 문제

1. 단기 기억과 장기 기억은 각각 무엇을 담당하는가?
2. 1-6·1-7에서 확인한 두 오류는 각각 그래프의 어느 단계에서 드러났는가?
3. Part 2와 Part 2의 도메인 반복에서, 단기·장기 기억 구조가 도메인이 달라도 그대로 통했는가?
4. `get_state`로 무엇을 확인할 수 있는가?
5. `hybrid_coach`에서 M1·M3·M4 각각에서 가져온 요소는 무엇인가?